# SurviveCity v1 — eval against extended LoRA

Training completed and the 20-step LoRA is on the Hub at
`noanya/zombiee-v1-extended`. This notebook just runs eval — no training, no
re-training, no risk to the trained checkpoints.

**Why this exists separately**: the train notebook crashed mid-eval because
the trained model learned to emit broadcast messages slightly longer than
v1's 40-char `SurviveAction.message` cap (it tried `"A2 unusually hungry,
potential infection?"` — 41 chars). That's actually a *good* learned
behaviour — the model is doing theory-of-mind reasoning — but the env's
strict pydantic schema rejects it. We patch `eval.py` inline to truncate
messages to 40 chars before they hit the env.

Time budget on Kaggle T4:
  - 30 baseline episodes (random policy): ~5 minutes
  - 30 trained episodes (LLM, ~3s per action × ~50 actions × 30 ep): ~75 min
  - Total: ~80 minutes


## 0. Bind to a single GPU BEFORE any imports

In [2]:
import os, sys, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("python:", sys.version.split()[0])


python: 3.12.12


## 1. Tokens (Kaggle Secrets → env)

In [3]:
HF_TOKEN = None
GITHUB_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    try: HF_TOKEN = _sec.get_secret("HF_TOKEN")
    except Exception: pass
    try: GITHUB_TOKEN = _sec.get_secret("GITHUB_TOKEN")
    except Exception: pass
except ImportError:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
GITHUB_TOKEN = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN")
assert HF_TOKEN, "Set HF_TOKEN in Kaggle Secrets before running."
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
print(f"HF_TOKEN     set ({len(HF_TOKEN)} chars, prefix={HF_TOKEN[:6]}...)")
print(f"GITHUB_TOKEN {'set' if GITHUB_TOKEN else 'absent (fine if repo is public)'}")


HF_TOKEN     set (37 chars, prefix=hf_jrZ...)
GITHUB_TOKEN set


## 2. Pin torch cu121 FIRST

In [4]:
import subprocess
def pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print(">>", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout: print(r.stdout[-1500:])
    if r.stderr: print("STDERR:", r.stderr[-800:])
    if check and r.returncode != 0: raise RuntimeError(f"pip failed (rc={r.returncode})")
    return r

if "torch" in sys.modules:
    raise RuntimeError("torch already imported — restart kernel.")

pip("install", "-q",
    "--index-url", "https://download.pytorch.org/whl/cu121",
    "--upgrade-strategy=only-if-needed",
    "torch==2.5.1+cu121", "torchvision==0.20.1+cu121", "torchaudio==2.5.1+cu121")


>> /usr/bin/python3 -m pip install -q --index-url https://download.pytorch.org/whl/cu121 --upgrade-strategy=only-if-needed torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 86.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 68.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 66.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 43.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 84.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 33.4 MB/s eta 0:00:00
  

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '--index-url', 'https://download.pytorch.org/whl/cu121', '--upgrade-strategy=only-if-needed', 'torch==2.5.1+cu121', 'torchvision==0.20.1+cu121', 'torchaudio==2.5.1+cu121'], returncode=0, stdout='     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.3 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 86.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 68.7 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 66.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 43.2 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 84.5 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00\n     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 1.3 MB/s eta 0:00:00

## 3. Pinned dep stack (same as the train notebook)

In [5]:
# All pinned packages that transformers 4.46.3 has range constraints on.
pip("install", "-q", "--no-deps", "--force-reinstall",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "huggingface_hub==0.27.1",
    "peft==0.13.2",
    "trl==0.15.2",
    "bitsandbytes==0.43.3",
    "accelerate==1.1.1",
    "datasets==3.1.0")
# Mergekit OFF — trl 0.15+ skips the import when it's absent.
pip("uninstall", "-y", "mergekit", check=False)
pip("install", "-q", "tensorboard", "tbparse", "psutil", "pydantic")
pip("uninstall", "-y", "torchao", check=False)

# transformers 4.46 removed TRANSFORMERS_CACHE; some optional imports still use it.
import sys as _sys
for _m in [m for m in _sys.modules if m.startswith("transformers") or m.startswith("huggingface_hub")]:
    del _sys.modules[_m]
import transformers.utils.hub as _hub
if not hasattr(_hub, "TRANSFORMERS_CACHE"):
    _hub.TRANSFORMERS_CACHE = getattr(_hub, "HF_HUB_CACHE", None) or os.path.expanduser("~/.cache/huggingface/hub")


>> /usr/bin/python3 -m pip install -q --no-deps --force-reinstall transformers==4.46.3 tokenizers==0.20.3 huggingface_hub==0.27.1 peft==0.13.2 trl==0.15.2 bitsandbytes==0.43.3 accelerate==1.1.1 datasets==3.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 30.2 MB/s eta 0:00:00

>> /usr/bin/python3 -m pip uninstall -y mergekit
STDERR: WARNING: S

## 4. Verify pins + GPU + deep imports

In [6]:
import importlib.metadata as _m
expected = {"transformers":"4.46.3","tokenizers":"0.20.3",
            "huggingface_hub":"0.27.1","peft":"0.13.2","trl":"0.15.2",
            "bitsandbytes":"0.43.3","accelerate":"1.1.1","datasets":"3.1.0"}
bad = []
for pkg, want in expected.items():
    try:
        got = _m.version(pkg)
        ok = got == want
        print(f"  {pkg:15s} want={want:10s} got={got:10s} {'OK' if ok else 'MISMATCH'}")
        if not ok: bad.append((pkg, want, got))
    except _m.PackageNotFoundError:
        print(f"  {pkg:15s} NOT INSTALLED"); bad.append((pkg, want, "missing"))
assert not bad, f"pinned versions didn't take: {bad}"

import torch
print(f"\n  torch={torch.__version__}  cuda={torch.cuda.is_available()}  cuda_ver={torch.version.cuda}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. Set Kaggle Settings → Accelerator → GPU T4 x1.")
print(f"  GPU: {torch.cuda.get_device_name(0)} cc={torch.cuda.get_device_capability(0)}")
free, total = torch.cuda.mem_get_info(0)
print(f"  VRAM free={free/1e9:.1f}/{total/1e9:.1f} GB")

# Real dep check
print("\n  importing transformers, peft, trl + GRPOTrainer...")
import transformers, peft, trl
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
print(f"  transformers {transformers.__version__}  peft {peft.__version__}  trl {trl.__version__}")
print("  full dep stack imports cleanly")


  transformers    want=4.46.3     got=4.46.3     OK
  tokenizers      want=0.20.3     got=0.20.3     OK
  huggingface_hub want=0.27.1     got=0.27.1     OK
  peft            want=0.13.2     got=0.13.2     OK
  trl             want=0.15.2     got=0.15.2     OK
  bitsandbytes    want=0.43.3     got=0.43.3     OK
  accelerate      want=1.1.1      got=1.1.1      OK
  datasets        want=3.1.0      got=3.1.0      OK

  torch=2.5.1+cu121  cuda=True  cuda_ver=12.1
  GPU: Tesla T4 cc=(7, 5)
  VRAM free=15.5/15.6 GB

  importing transformers, peft, trl + GRPOTrainer...


2026-04-26 01:52:07.512616: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777168327.726955      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777168327.791099      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777168328.296600      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777168328.296649      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777168328.296652      55 computation_placer.cc:177] computation placer alr

  transformers 4.46.3  peft 0.13.2  trl 0.15.2
  full dep stack imports cleanly


## 5. Clone repo + cd into v1

In [7]:
import base64
os.environ["GIT_TERMINAL_PROMPT"] = "0"
os.environ["GIT_ASKPASS"] = "/bin/echo"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/tmp"
REPO_DIR = os.path.join(WORK, "zombiee")
REPO_URL = os.environ.get("REPO_URL", "https://github.com/SirjanSingh/zombiee.git")

if os.path.isdir(REPO_DIR):
    print(f"repo already cloned at {REPO_DIR}")
    if GITHUB_TOKEN:
        auth = "Authorization: Basic " + base64.b64encode(
            f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
        r = subprocess.run(
            ["git", "-c", f"http.extraheader={auth}", "-C", REPO_DIR, "pull", "--ff-only"],
            capture_output=True, text=True)
        print("  pull:", r.stdout.strip() or r.stderr.strip()[:200] or "(no change)")
    else:
        print("  skipping pull (no GITHUB_TOKEN; existing checkout is fine).")
else:
    cmd = ["git", "clone", "--depth=1", REPO_URL, REPO_DIR]
    if GITHUB_TOKEN:
        auth = "Authorization: Basic " + base64.b64encode(
            f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
        cmd = ["git", "-c", f"http.extraheader={auth}"] + cmd[1:]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout); print(r.stderr)
    assert r.returncode == 0, "git clone failed"
V1 = os.path.join(REPO_DIR, "v1")
os.chdir(V1)
sys.path.insert(0, V1)
print("CWD:", os.getcwd())



Cloning into '/kaggle/working/zombiee'...

CWD: /kaggle/working/zombiee/v1


## 6. Download the trained LoRA from `noanya/zombiee-v1-extended`

This is the 20-step LoRA we just trained. Snapshot-download it locally so
v1's eval.py can load it via `--lora-path`.


In [8]:
from huggingface_hub import snapshot_download, login
login(token=HF_TOKEN, add_to_git_credential=False)

EXTENDED_REPO = os.environ.get("EXTENDED_REPO", "noanya/zombiee-v1-extended")
LOCAL_LORA = os.path.join(WORK, "lora_v1_extend")

print(f"Downloading {EXTENDED_REPO} → {LOCAL_LORA}...")
snapshot_download(
    repo_id=EXTENDED_REPO,
    local_dir=LOCAL_LORA,
    # Skip the heavy checkpoint-* dirs; eval only needs the root adapter.
    ignore_patterns=["checkpoint-*/*", "runs/*", "eval_results/*"],
)
files = os.listdir(LOCAL_LORA)
print(f"\n  files at {LOCAL_LORA}:")
for f in sorted(files): print(f"    {f}")
required = {"adapter_config.json", "adapter_model.safetensors"}
missing = required - set(files)
assert not missing, f"missing required adapter files: {missing}"
print("\n  LoRA ready for eval.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

...

.gitattributes: 0.00B [00:00, ?B/s]

training_args.bin:   0%|          | 0.00/5.75k [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]


  files at /kaggle/working/lora_v1_extend:
    .cache
    .gitattributes
    README.md
    adapter_config.json
    adapter_model.safetensors
    added_tokens.json
    merges.txt
    special_tokens_map.json
    tokenizer.json
    tokenizer_config.json
    training_args.bin
    vocab.json

  LoRA ready for eval.


## 7. Inline-patch v1/training/eval.py to truncate broadcast messages

The trained model learned to emit broadcasts slightly over the 40-char cap
(specifically `"A2 unusually hungry, potential infection?"` = 41 chars). v1's
env strictly rejects messages >40 chars via pydantic. The fix: truncate
the message in the LLM action fn before it reaches the env.


In [ ]:
import re, pathlib as _pl
_ep = _pl.Path("training/eval.py")
_src = _ep.read_text()

if "MESSAGE_MAX = 40" in _src:/
    print("eval.py already patched.")
else:
    # Insert a small truncate after the `parsed = ...` line in make_llm_action_fn.
    # Specifically: after a successful JSON parse, if action_type == "broadcast"
    # and message is set, truncate to 40 chars.
    # Find a stable anchor: the function-end of make_llm_action_fn before the
    # `return llm_action` line.
    _new = _src
    if 'return llm_action' in _new and "MESSAGE_MAX" not in _new:
        # Add a tiny truncation guard inline. We patch by looking for
        # `parsed = _json.loads(...)` style line OR by wrapping the return at
        # the end of llm_action(). The robust approach: monkeypatch the env
        # at runtime — but cleaner is to fix make_llm_action_fn.
        #
        # Direct replacement: find `def llm_action(...)` body and add a guard
        # right before the function returns the parsed dict.
        _new = _new.replace(
            "def make_llm_action_fn(model, tokenizer):",
            "MESSAGE_MAX = 40  # SurviveAction.message hard cap (added by eval-extend notebook)\n"
            "def make_llm_action_fn(model, tokenizer):",
            1,
        )
        # Add truncation by wrapping the returned llm_action with a decorator-
        # style post-process. Simpler: search for the line that returns parsed
        # in llm_action() and truncate the message before that.
        # In v1's eval.py this looks like `return parsed` or similar inside llm_action.
        # We'll do a robust global truncate: any time llm_action returns a dict,
        # if its action_type == "broadcast", truncate message.
        # Use AST-free approach: replace the return statement of llm_action.
        _patched = re.sub(
            r"(\bdef llm_action\b.*?)(\n\s+return\s+\{[^}]*\})",
            lambda m: m.group(1) + m.group(2) + "  # noqa",
            _new, flags=re.DOTALL,
        )
        # That regex may not match v1's exact code; simpler/more reliable: append
        # a post-processing wrapper inside make_llm_action_fn just before `return llm_action`.
        _patched = re.sub(
            r"(\n\s+)return llm_action\b",
            r"""\1_orig_llm_action = llm_action
\1def _truncated_llm_action(agent_id, obs):
\1    a = _orig_llm_action(agent_id, obs)
\1    if isinstance(a, dict) and a.get("action_type") == "broadcast":
\1        m = a.get("message")
\1        if isinstance(m, str) and len(m) > MESSAGE_MAX:
\1            a["message"] = m[:MESSAGE_MAX]
\1    return a
\1return _truncated_llm_action""",
            _patched, count=1,
        )
        _ep.write_text(_patched)
        print("eval.py patched: broadcast messages now truncated to 40 chars before env.step.")

# Reload eval module so the patch takes effect
import importlib
if "training.eval" in sys.modules:
    del sys.modules["training.eval"]
import training.eval as _eval_mod
print("training.eval reloaded.")


eval.py patched: broadcast messages now truncated to 40 chars before env.step.
training.eval reloaded.


## 8. Pre-flight: free GPU memory, sanity check

In [10]:
import gc
gc.collect()
import torch
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info(0)
print(f"VRAM before eval: free={free/1e9:.1f}/{total/1e9:.1f} GB")
if free / 1e9 < 8:
    print("  WARN: <8 GB free — eval may OOM. Restart kernel and re-run cells if needed.")


VRAM before eval: free=15.5/15.6 GB


## 9. Run eval — 60 episodes (30 baseline + 30 trained)

Hyperparameters identical to the original v1 Colab eval, just with the larger
sample for tighter confidence intervals. Output goes to
`/kaggle/working/lora_v1_extend/plots/`.

Estimated time: ~80 minutes total (5 baseline + 75 trained).


In [11]:
import importlib, training.eval as _eval_mod
importlib.reload(_eval_mod)

EVAL_ARGS = [
    "--lora-path", LOCAL_LORA,
    "--model-name", "Qwen/Qwen2.5-3B-Instruct",
    "--episodes", "60",
    "--seed", "42",
    "--plots-dir", os.path.join(LOCAL_LORA, "plots"),
]
sys.argv = ["eval.py"] + EVAL_ARGS
print("Launching eval:")
for a in EVAL_ARGS: print("  ", a)
print()
try:
    _eval_mod.main()
except SystemExit:
    pass


INFO:survivecity.eval:Running baseline evaluation (60 episodes)...


Launching eval:
   --lora-path
   /kaggle/working/lora_v1_extend
   --model-name
   Qwen/Qwen2.5-3B-Instruct
   --episodes
   60
   --seed
   42
   --plots-dir
   /kaggle/working/lora_v1_extend/plots



INFO:survivecity_env.env:[START] episode=1 infected=A2 seed=670487
INFO:survivecity_env.env:[STEP] ep=1 step=0 agent=A0 action=eat reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=0 agent=A1 action=wait reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=1 agent=A2 action=wait reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=1 agent=A0 action=move_left reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=1 agent=A1 action=move_down reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=2 agent=A2 action=wait reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=2 agent=A0 action=move_left reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=2 agent=A1 action=move_down reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=3 agent=A2 action=move_down reward=0.0100 raw=0.0050 done=False


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

INFO:survivecity.eval:Loading LoRA adapter from: /kaggle/working/lora_v1_extend
INFO:survivecity.eval:✅ Trained model loaded successfully
INFO:survivecity.eval:Running trained evaluation (60 episodes)...
INFO:survivecity_env.env:[START] episode=1 infected=A2 seed=261630
INFO:survivecity_env.env:[STEP] ep=1 step=0 agent=A0 action=move_right reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=0 agent=A1 action=wait reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=1 agent=A2 action=wait reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=1 agent=A0 action=wait reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=1 agent=A1 action=move_left reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=2 agent=A2 action=move_right reward=0.0100 raw=0.0050 done=False
INFO:survivecity_env.env:[STEP] ep=1 step=2 agent=A0 action=wait reward=0.0100 raw=0.0050 done=False
INFO:

## 10. Push eval artefacts to `noanya/zombiee-v1-extended/eval_results`

In [15]:
from huggingface_hub import upload_folder

eval_dir = os.path.join(LOCAL_LORA, "plots")
if os.path.isdir(eval_dir):
    upload_folder(
        folder_path=eval_dir,
        path_in_repo="eval_results",
        repo_id=EXTENDED_REPO,
        repo_type="model",
        commit_message="step-20 eval (n_t=30 vs n_b=30) with broadcast-truncate fix",
    )
    print(f"  uploaded {eval_dir} -> {EXTENDED_REPO}/eval_results")
    print(f"  view: https://huggingface.co/{EXTENDED_REPO}/tree/main/eval_results")
else:
    print(f"  WARN: no plots dir at {eval_dir} — eval may not have produced output.")


No files have been modified since last commit. Skipping to prevent empty commit.


  uploaded /kaggle/working/lora_v1_extend/plots -> noanya/zombiee-v1-extended/eval_results
  view: https://huggingface.co/noanya/zombiee-v1-extended/tree/main/eval_results


## 11. Print headline numbers ready to paste into v1.tex Table 2

In [13]:
import json, glob
eval_jsons = sorted(glob.glob(os.path.join(LOCAL_LORA, "plots", "*.json")))
if not eval_jsons:
    print("no eval json found")
else:
    latest = eval_jsons[-1]
    print(f"=== {os.path.basename(latest)} ===")
    with open(latest) as f:
        d = json.load(f)
    print(json.dumps({
        "baseline": d.get("baseline_summary"),
        "trained":  d.get("trained_summary"),
        "delta":    d.get("delta"),
    }, indent=2))
    print()
    print("If trained 'survival_rate' > 0 and 'mean_total_reward' > baseline,")
    print("update report/v1/v1.tex Table 2 to reference these numbers and")
    print(f"point figures at {EXTENDED_REPO}/eval_results/.")


no eval json found
